In [1]:
! uv pip install langchain openai tiktoken rapidocr-onnxruntime python-dotenv langchain-community

Using Python 3.12.7 environment at: C:\Users\asker\Documents\GenAIProjects\RAG_LLMOPS\.venv
Resolved 72 packages in 1.53s
 Downloaded shapely
 Downloaded pydantic-core
 Downloaded sqlalchemy
 Downloaded pillow
 Downloaded openai
 Downloaded numpy
 Downloaded rapidocr-onnxruntime
 Downloaded langchain-community
 Downloaded onnxruntime
 Downloaded sympy
 Downloaded opencv-python
Prepared 68 packages in 30.23s
Installed 68 packages in 1.98s
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.5
 + aiosignal==1.4.0
 + annotated-types==0.7.0
 + anyio==4.13.0
 + attrs==26.1.0
 + certifi==2026.2.25
 + charset-normalizer==3.4.7
 + dataclasses-json==0.6.7
 + distro==1.9.0
 + flatbuffers==25.12.19
 + frozenlist==1.8.0
 + greenlet==3.4.0
 + h11==0.16.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + httpx-sse==0.4.3
 + idna==3.11
 + jiter==0.14.0
 + jsonpatch==1.33
 + jsonpointer==3.1.1
 + langchain==1.2.15
 + langchain-classic==1.0.3
 + langchain-community==0.4.1
 + langchain-core==1.2.30
 + langchain-text-splitt

In [3]:
from langchain_community.document_loaders import TextLoader


In [7]:
!pip install pypdf

In [9]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

# Initialize the loader pointing to your data directory
loader = PyPDFDirectoryLoader("../data")

# Load all PDFs
documents = loader.load()

# Let's check how many pages were loaded
print(f"Successfully loaded {len(documents)} pages from the PDFs.")


Successfully loaded 414 pages from the PDFs.


In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize the text splitter
# You can tweak chunk_size and chunk_overlap based on the specific LLM / embeddings you plan to use
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # Max characters per chunk
    chunk_overlap=200,    # How many characters chunks should overlap to preserve context boundaries
    length_function=len,
    add_start_index=True  # Optional metadata tracking original starting position
)

# Apply the splitter to the documents you just loaded earlier
chunks = text_splitter.split_documents(documents)

# Let's verify what happened
print(f"Split {len(documents)} original documents into {len(chunks)} overlapping chunks.")

# Quick preview of the first chunk
print("\n--- First Chunk Sample ---")
print(chunks[0].page_content[:300] + "...")


Split 414 original documents into 1306 overlapping chunks.

--- First Chunk Sample ---
1 | P a g e  
 
ENTSO-E AISBL •  Avenue Cortenbergh 100  • 1000 Brussels • Belgium • Tel +32 2 741 09 50  • Fax +32 2 741 09 51  
 
 
 
 
 
 
   
 
 Network Code on 
Load-Frequency Control and Reserves 
 
 
28 June 2013 
 
  
Notice 
This document reflects the status of the work of Transmission Syst...


In [11]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Initialize HuggingFace embeddings
# "all-MiniLM-L6-v2" is an excellent, lightweight, and fast local embedding model.
# NOTE: When you first run this, it will automatically download the model weights (~80MB).
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Turn our chunked documents into a FAISS vector database
print("Building FAISS vector store... This might take a minute depending on document length.")
vectorstore = FAISS.from_documents(chunks, embeddings)

print(f"Successfully created vector store with {vectorstore.index.ntotal} vectors!")

# 3. (Optional but highly recommended) Save the database locally so you don't have to rebuild it next time
vectorstore.save_local("../data/faiss_index")


c:\Users\asker\Documents\GenAIProjects\RAG_LLMOPS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3150.28it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Building FAISS vector store... This might take a minute depending on document length.
Successfully created vector store with 1306 vectors!


In [12]:
# Write a query related to the energy documents you uploaded
query = "What are the key impacts of renewable energy sources on the grid transmission system?"

# Perform a similarity search
# k=3 tells it to retrieve exactly the top 3 most relevant chunks
results = vectorstore.similarity_search(query, k=3)

print(f"Found {len(results)} relevant results for your query: '{query}'\n")

# Loop through and print the results nicely
for i, result in enumerate(results, 1):
    print(f"--- MATCH {i} ---")
    
    # Extract metadata like the source PDF and page number
    source_file = result.metadata.get('source', 'Unknown Source')
    page_num = result.metadata.get('page', 'Unknown Page')
    
    print(f"Source: {source_file} (Page {page_num})")
    
    # Print the first 500 characters of the text chunk to see what matched!
    print(f"Content: {result.page_content[:500]}...\n")


Found 3 relevant results for your query: 'What are the key impacts of renewable energy sources on the grid transmission system?'

--- MATCH 1 ---
Source: ..\data\Germany2025.pdf (Page 50)
Content: , based on the 
findings of the 2023 Platform Climate-Neutral Electricity System. The options laid out 
in the report are based on four fields of action: 1)  the renewables investment 
framework; 2)  the investment framework for dispatchable capacity; 3)  locational 
signals; and 4) demand flexibility.  
Transmission grid 
Insufficient power grid capacity remains a notable impediment to Germany’s energy 
transition. A key solution to alleviating its regional power imbalances and addressing 
grid...

--- MATCH 2 ---
Source: ..\data\2024-18_EN_JAW24_Presentation_250110.pdf (Page 30)
Content: Outlook→Power sector as driving force: The expansion of renewable energy and electricity grids has gained significant momentum. →This means that rapidly increasing quantities of renewable electricity will b

In [21]:
import os
from dotenv import load_dotenv

# 1. Load the variables from the hidden .env file into this notebook session
# The override=True ensures that it refreshes the variables if you ran a previous cell
load_dotenv(override=True)

# Let's double check that it actually found the key!
if not os.getenv("GROQ_API_KEY"):
    print("WARNING: GROQ_API_KEY was not found in the environment!")

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

# 2. Initialize the Groq LLM
# It will now successfully find the key that was just loaded!
llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0)


# 3. Build the RAG Prompt Template
template = """You are an expert Energy Analyst and AI assistant.
Please answer the user's technical question based STRICTLY and solely on the provided context retrieved from our energy documents.
If the answer is not contained in the context, simply say "I cannot find the answer to this in the provided documents" - do not hallucinate outside information.

Context from Documents:
{context}

User Question:
{question}

Expert Answer:"""

prompt = ChatPromptTemplate.from_template(template)

print("Groq LLM and Prompt Template successfully initialized!")


Groq LLM and Prompt Template successfully initialized!


In [23]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.runnables import RunnablePassthrough
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.output_parsers import StrOutputParser

# 1. Update your original prompt format to accept chat history
system_template = """You are an expert Energy Analyst and AI assistant.
Answer the user's technical question based STRICTLY and solely on the provided context retrieved from our energy documents.

Context from Documents:
{context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_template),
    MessagesPlaceholder(variable_name="chat_history"), # Memory gets injected 
    ("human", "{question}")
])

# 2. Build the Retriever and formatting function
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 3. YOUR EXACT CHAIN LOGIC: (Context + Question) -> Prompt -> LLM -> Parser
pure_lcel_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 4. Create the memory dictionary to track sessions
chat_history_store = {}

def get_session_history(session_id: str):
    if session_id not in chat_history_store:
        chat_history_store[session_id] = ChatMessageHistory()
    return chat_history_store[session_id]

# 5. Wrap your pure chain so it intercepts and loads chat history!
conversational_rag_chain = RunnableWithMessageHistory(
    pure_lcel_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)

print("Your custom LCEL Memory Chain is successfully built!")


Your custom LCEL Memory Chain is successfully built!


In [24]:
from operator import itemgetter
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.runnables import RunnablePassthrough
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.output_parsers import StrOutputParser

# 1. Prompt exactly as before
system_template = """You are an expert Energy Analyst and AI assistant.
Answer the user's technical question based STRICTLY and solely on the provided context retrieved from our energy documents.

Context from Documents:
{context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_template),
    MessagesPlaceholder(variable_name="chat_history"), # Memory injected here
    ("human", "{question}")
])

# 2. Retriever and Formatter
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 3. FIXED PURE LCEL CHAIN: 
# `assign` perfectly preserves {question} and {chat_history} from memory manager, and just adds {context}!
pure_lcel_chain = (
    RunnablePassthrough.assign(
        context = itemgetter("question") | retriever | format_docs
    )
    | prompt
    | llm
    | StrOutputParser()
)

# 4. Memory Store
chat_history_store = {}

def get_session_history(session_id: str):
    if session_id not in chat_history_store:
        chat_history_store[session_id] = ChatMessageHistory()
    return chat_history_store[session_id]

# 5. Wrap it in history!
conversational_rag_chain = RunnableWithMessageHistory(
    pure_lcel_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)

print("Your custom LCEL Memory Chain is successfully rebuilt!")


Your custom LCEL Memory Chain is successfully rebuilt!


In [25]:
# Question 1: Establishing the subject
print("--- Question 1 ---")
resp1 = conversational_rag_chain.invoke(
    {"question": "What is the Load-Frequency Control Reserve (LFCR) according to the documents?"},
    config={"configurable": {"session_id": "energy_session"}}
)
print("Groq:\n", resp1, "\n")


# Question 2: Testing Memory ("it")
print("--- Question 2 ---")
resp2 = conversational_rag_chain.invoke(
    {"question": "What is its main purpose in the grid?"},
    config={"configurable": {"session_id": "energy_session"}}
)
print("Groq:\n", resp2, "\n")


# Question 3: Testing Deeper Memory ("it")
print("--- Question 3 ---")
resp3 = conversational_rag_chain.invoke(
    {"question": "Are there specific technical rules or capacity requirements mentioned for it?"},
    config={"configurable": {"session_id": "energy_session"}}
)
print("Groq:\n", resp3, "\n")


--- Question 1 ---
Groq:
 The Load-Frequency Control Reserve (LFCR) is not explicitly mentioned in the provided documents. However, the documents do mention the following related terms:

- FCR (Frequency Containment Reserve): aims at containing the System Frequency deviation after an incident within a pre-defined range.
- FRR (Frequency Restoration Reserve): aims at restoring the System Frequency to its Nominal Frequency of 50 Hz.
- RR (Replacement Reserve): replaces the activated reserves to restore the available reserves in the system or for economic optimisation.

It appears that the Load-Frequency Control Reserve (LFCR) might be a broader concept that encompasses FCR, FRR, and RR, but it is not explicitly defined in the provided documents. 

--- Question 2 ---
Groq:
 The Load-Frequency Control Reserve (LFCR) is not explicitly mentioned in the provided documents. However, based on general knowledge, the main purpose of Load-Frequency Control Reserve (LFCR) in the grid is to:

- Main